# Se gradient descent og backpropagation i arbeid

**Du skal kunne:** skille mellom feilmål, gradient og oppdatering av vekter.
Vi starter med én justerbar verdi, og lærer deretter en rett linje fra kunstige data.
Alt er lite nok til CPU. Sammenlign med videoene dere har sett.


## Gjør arbeidsbenken klar

`import` henter verktøy; `as` gir et kortere navn. `Path` håndterer filstier.
Vi finner kursmappen og legger `startkode` i Pythons søkesti, slik at samme kode
virker fra VS Code og JupyterLab. `sett_seed(42)` gir et fast startpunkt for
slumptall; det gjør sammenligning lettere, men lover ikke identiske tall på ulike
maskiner. `velg_enhet()` velger NVIDIA-GPU når CUDA er tilgjengelig, ellers CPU.
En **enhet** er stedet beregningene skjer. Modell og data må ligge på samme enhet.

**Før kjøring:** Gjett om utskriften blir `cpu` eller `cuda`.
**Etter kjøring:** Kontroller at det stemmer med installasjonssjekken.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

# Finn kursmappen både fra en notatbok og fra prosjektroten.
her = Path.cwd().resolve()
kandidater = [her, *her.parents, her / "05-dypere-maskinlaering"]
kursmappe = next(p for p in kandidater if (p / "startkode" / "kursverktoy.py").exists())
if str(kursmappe / "startkode") not in sys.path:
    sys.path.insert(0, str(kursmappe / "startkode"))
from kursverktoy import sett_seed, velg_enhet
sett_seed(42)
enhet = velg_enhet()
print("PyTorch:", torch.__version__, "| beregner på:", enhet)


## En bakke med ett bunnpunkt

Vi vil gjøre `(vekt - 3)²` minst mulig. Det er en skål med bunn ved 3.
`requires_grad=True` ber PyTorch huske beregningene. `backward()` følger kjederegelen
bakover og beregner den deriverte. Den endrer **ikke** vekten.
`grad` forteller helningen: positiv helning betyr at et lite steg mot venstre senker feilen.

**Regn først:** Ved vekt 0 er feilen 9 og gradienten `2 × (0 - 3) = -6`.


In [ ]:
vekt = torch.tensor(0.0, requires_grad=True)
feil = (vekt - 3) ** 2
feil.backward()
print("Feil:", feil.item(), "gradient:", vekt.grad.item(), "vekt:", vekt.item())


## Optimizer tar steget

`SGD` er en optimizer som trekker læringsrate ganger gradient fra vekten.
Listen `[vekt]` sier hvilke verdier den skal justere. Med læringsrate 0,1 blir første
steg `0 - 0,1 × (-6) = 0,6`. `zero_grad()` fjerner forrige gradient før neste regnestykke;
ellers summerer PyTorch gradientene. `.item()` gjør én tensorverdi til et vanlig Python-tall.
Historikken er en liste vi kan tegne.

**Forutsi:** Vil vekten nærme seg 3 nedenfra eller ovenfra?


In [ ]:
vekt = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([vekt], lr=0.1)
historikk = []
for steg in range(30):
    optimizer.zero_grad()
    feil = (vekt - 3) ** 2
    feil.backward()
    optimizer.step()
    historikk.append((vekt.item(), feil.item()))
print("Siste vekt:", vekt.item())
plt.plot([rad[1] for rad in historikk])
plt.xlabel("Steg")
plt.ylabel("Feil før oppdatering")
plt.show()


## Lær både stigning og startverdi

Nå lager vi observasjoner rundt `y = 2x + 1`. Støy gjør at punktene ikke ligger helt
på linjen. `linspace` lager jevnt fordelte x-verdier; `unsqueeze(1)` gir formen
`[100, 1]`: hundre eksempler med én egenskap hver. `nn.Linear(1, 1)` har to lærbare
parametre: stigning (weight) og startverdi (bias). `MSELoss` er gjennomsnittet av
kvadrerte avvik, et egnet feilmål for dette regresjonsforsøket.

`parameters()` henter begge parametrene. Samme tre trinn går igjen: beregn gradient,
nullstill gammel gradient før beregningen, og la optimizer oppdatere. Her bruker
vi hele datasettet i hvert steg, derfor er dette batch gradient descent.


In [ ]:
from torch import nn
sett_seed(42)
x = torch.linspace(-1, 1, 100).unsqueeze(1)
y = 2 * x + 1 + 0.15 * torch.randn_like(x)
modell = nn.Linear(1, 1)
optimizer = torch.optim.SGD(modell.parameters(), lr=0.1)
feilmaal = nn.MSELoss()
for steg in range(150):
    optimizer.zero_grad()
    prediksjon = modell(x)
    feil = feilmaal(prediksjon, y)
    feil.backward()
    optimizer.step()
print("Stigning:", modell.weight.item(), "startverdi:", modell.bias.item())
with torch.no_grad():
    linje = modell(x)
plt.scatter(x[:, 0], y[:, 0], s=10, label="Observasjoner")
plt.plot(x[:, 0], linje[:, 0], color="red", label="Lært linje")
plt.legend()
plt.show()


`no_grad()` slår av registrering av beregninger når vi bare skal se på modellen.
Det sparer arbeid og minne. Det er forskjellig fra å endre læringsraten til null.

## Eksperiment og refleksjon

Kjør skålforsøket på nytt med `lr=0.01`, `0.9` og `1.1`. Lag en ny vekt og optimizer
hver gang. Ved 0,01 går det sakte; ved 0,9 hopper vi frem og tilbake, men nærmer oss;
ved 1,1 vokser avstanden. Store steg kan føre oss opp av skålen.

Forklar med egne ord hvilken linje som gjør backpropagation og hvilken som endrer
vektene. **Fasit:** `backward()` beregner; `step()` oppdaterer. Sammenlign dette med
ordbruken i del 4: de to handlingene er separate.

**Neste:** [02-bilder-og-treningslokke.ipynb](02-bilder-og-treningslokke.ipynb).
